# Recompute `YrA` from the saved footprints

Driver for `caban.yra_recompute`. All the logic lives in `caban/yra_recompute.py`;
this notebook is the parameter cell, the queue, and the report
(`plans/yra_recompute_plan.md` §11.1).

**How to use it:** run the setup and discovery cells once, then alternate the
**preview** and **run** cells. Preview shows you what is next and cross-checks its
inputs without computing; run does the work and prints the report. Preview again
and it has already advanced to the next session — position is re-derived from what
is on disk each time, so a kernel restart costs you nothing and a finished session
is never redone.

**Order of execution** (§11.7): G10 `TFC_cond` was the §7.1 gate and has passed
(§12). The queue below picks up from there.

`YrA_recomputed.zarr` and its `YrA_recompute.json` sidecar are written beside the
`A`/`C`/`S` they come from, inside the session's `minian_crossreg*` folder; the existing
`YrA.zarr` there is never touched (§11.3).

In [ ]:
import os
import sys
import time

import dask
import pandas as pd

sys.path.insert(0, os.path.abspath(".."))
from caban import yra_recompute as yr

pd.set_option("display.max_rows", 200, "display.width", 220)

## Parameters

Nothing here is per-session. Every path a session needs — its videos, its
`minian_crossreg*`, its old `YrA.zarr` if it has one, its saved `Y` if it has one —
is resolved by discovery, and `del_frames` is looked up from the notebook corpus
(see `yr.NOTEBOOK_DEL_FRAMES`: across all 60 notebooks in the Minian mirror the only
session with frames dropped is a G25 one, outside this cohort). The derivation from
the saved `frame` coordinate still runs as an independent cross-check, and a
disagreement is a hard failure (§6.1).

`MINIAN_DIR_OVERRIDES` exists because discovery refuses to guess. G16's
`2022_01_26-TFC_test_B / 14_23_45-LT1` has two complete `minian_crossreg*` outputs
that disagree about both unit and frame count, and only one of them can belong to
this session: the videos and `timeStamps.csv` hold 13,923 frames, which is
`minian_crossreg3` (788 x 13923). `minian_crossreg2_crossreg3` is (883 x 17853) —
more frames than the session has, so it belongs to something else. Hence the
override below.

Picking wrong is not silently possible: `derive_deleted_frames` raises on a `frame`
coordinate that runs past the end of the videos, which is exactly what the other
folder does. The override says which one is meant; the guard enforces that it fits.

In [ ]:
DATA_ROOTS = None                   # auto: /Volumes/MINISCOPE if mounted, else the
                                    # two read-only BAK drives

MINIAN_DIR_OVERRIDES = {
    "G16/2022_01_26-TFC_test_B/14_23_45-LT1": "minian_crossreg3",
}

# Narrow the queue while working through a mouse or a session type. None = everything.
MOUSE_FILTER = None                 # e.g. ["G05", "G08"]
SESSION_FILTER = None               # substring of the session name, e.g. "TFC_cond"

N_WORKERS = 6
FRAME_CHUNK = yr.DEFAULT_FRAME_CHUNK
FORCE = False                       # recompute even a session whose sidecar says done

dask.config.set(scheduler="threads", num_workers=N_WORKERS)

## Discover the queue

A session is work exactly when its `Miniscope` folder has numbered `.avi` files and
one complete `minian_crossreg*`. The counts printed below account for everything
scanned, so what is *not* in the queue is visible rather than implied — the ~976
excluded folders are the HC and CNO sessions that were never put through CNMF-E.

The table is the queue. `done` comes from a cheap check (a complete sidecar with its
output beside it); the authoritative check re-hashes every input and runs inside
`run_item`, so a session that merely looks finished is still caught.

In [ ]:
items = yr.discover_sessions(
    roots=DATA_ROOTS,
    minian_dir_overrides=MINIAN_DIR_OVERRIDES,
)

if MOUSE_FILTER:
    items = [i for i in items if i.mouse in MOUSE_FILTER]
if SESSION_FILTER:
    items = [i for i in items if SESSION_FILTER in i.session]

status = yr.queue_status(items)
print(f"\n{int(status['done'].sum())} of {len(status)} done")
status

## Preview the next session

Re-run this cell to advance. It reports the resolved paths, the unit and frame
counts, and the two `del_frames` values side by side, then stops before the
replay.

In [ ]:
item = yr.next_pending(items)
if item is None:
    print("queue empty -- every session in this selection is done")
else:
    _ = yr.preview_item(item, frame_chunk=FRAME_CHUNK)

## Run it

~11 min for a conditioning session at `N_WORKERS = 6`, less for the shorter LT
sessions; about half of it is decoding `.avi` over USB. Writes
`YrA_recomputed.zarr` plus its `YrA_recompute.json` sidecar, then prints the report.

Hard assertions run inside the call, so reaching the report means `unit_id` matches
`S`'s elementwise and unsorted, shape and `frame` match `C`, and there are no NaN or
inf values.

**Read the `overlap none` row first.** Those units have `YrA_i` determined by the
replayed movie and their own footprint alone — no `C` at all — so a median r that is
not ~0.999999 means that session's `Y` did not replay, and the queue should stop
rather than roll on. The `>10 %` row sitting lower is the known `C`-version
difference of §12.4, not a replay problem. Sessions with no old `YrA` have no
comparison block at all and lean on the hard checks plus `corr(C, YrA_new)`.

In [ ]:
started = time.time()
record = yr.run_item(item, frame_chunk=FRAME_CHUNK, force=FORCE)
print(f"\nelapsed {time.time() - started:.1f} s")

## Or work through the rest unattended

Serial by design (§11.5) — each session is already memory- and I/O-bound through its
own dask graph. Roughly a day for all 130. `stop_on_error=True` means a failed hard
check halts the batch, which is the point of having hard checks; drop it to `False`
only for a sweep whose failures you intend to read afterwards in the returned list.

Left commented out: stepping is the default because the report is meant to be read.

In [ ]:
# records = yr.run_all(items, frame_chunk=FRAME_CHUNK, force=FORCE)
# yr.queue_status(items)